In [1]:
from PyPDF2 import PdfReader, PdfWriter, PdfMerger
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
import io

In [2]:
import fitz  # PyMuPDF
from PIL import Image

In [13]:
for y in reversed(range(gray.height)):
    print(y)

1223
1222
1221
1220
1219
1218
1217
1216
1215
1214
1213
1212
1211
1210
1209
1208
1207
1206
1205
1204
1203
1202
1201
1200
1199
1198
1197
1196
1195
1194
1193
1192
1191
1190
1189
1188
1187
1186
1185
1184
1183
1182
1181
1180
1179
1178
1177
1176
1175
1174
1173
1172
1171
1170
1169
1168
1167
1166
1165
1164
1163
1162
1161
1160
1159
1158
1157
1156
1155
1154
1153
1152
1151
1150
1149
1148
1147
1146
1145
1144
1143
1142
1141
1140
1139
1138
1137
1136
1135
1134
1133
1132
1131
1130
1129
1128
1127
1126
1125
1124
1123
1122
1121
1120
1119
1118
1117
1116
1115
1114
1113
1112
1111
1110
1109
1108
1107
1106
1105
1104
1103
1102
1101
1100
1099
1098
1097
1096
1095
1094
1093
1092
1091
1090
1089
1088
1087
1086
1085
1084
1083
1082
1081
1080
1079
1078
1077
1076
1075
1074
1073
1072
1071
1070
1069
1068
1067
1066
1065
1064
1063
1062
1061
1060
1059
1058
1057
1056
1055
1054
1053
1052
1051
1050
1049
1048
1047
1046
1045
1044
1043
1042
1041
1040
1039
1038
1037
1036
1035
1034
1033
1032
1031
1030
1029
1028
1027
1026
1025
1024


In [2]:
from pdf2image import convert_from_path
import numpy as np

# Step 1: Convert first page of PDF to image
pages = convert_from_path(r"C:\Users\Matthew.Vaughn\Desktop\Mishandled Claims Test.pdf", dpi=300)  # increase dpi for accuracy
img = pages[0].convert("RGB")

# Step 2: Convert to NumPy array
arr = np.array(img)

# Step 3: Define "white"
# (tolerating near-white pixels)
threshold = 245
is_white = np.all(arr > threshold, axis=2)  # True if pixel is (near) white

# Step 4: Scan from bottom up
height = arr.shape[0]  # total height in pixels
lowest_nonwhite = None

for row in range(height - 1, -1, -1):  # bottom → top
    if not np.all(is_white[row]):      # found a row with non-white pixels
        lowest_nonwhite = row
        break

print("Lowest non-white pixel row (from top):", lowest_nonwhite)
print("Height from bottom:", height - lowest_nonwhite)


PDFInfoNotInstalledError: Unable to get page count. Is poppler installed and in PATH?

In [20]:
pdf_path = r"C:\Users\Matthew.Vaughn\Desktop\Mishandled Claims Test.pdf"
doc = fitz.open(pdf_path)

for page_num, page in enumerate(doc, start=1):
    # Render page to a high-res image
    zoom = 2  # 2x resolution
    mat = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, alpha=False)
    
    # Convert to PIL image
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    
    # Convert to grayscale
    gray = img.convert("L")
    pixels = gray.load()
    
    # Scan from bottom to top to find first non-white row
    bottom = 0
    for y in range(gray.height):
        for x in range(gray.width):
            if pixels[x, y] < 270:  # threshold for "non-white"
                bottom = y
                break
        if bottom:
            break
    
    # Convert back to PDF points (top-left origin)
    page_height_points = page.mediabox.height
    bottom_in_points = page_height_points * (bottom / gray.height)
    
    print(f"Page {page_num} bottom of content (points): {bottom_in_points}")

    print(page_height_points)
    print(bottom_in_points)
    print(bottom)
    print(gray.height)
    print("-----")
    print(gray.height - page_height_points)


Page 1 bottom of content (points): 0.5
612.0
0.5
1
1224
-----
612.0


In [5]:
# Function to create a header overlay
def create_header(page_width, page_height, left_text="DRAFT - CONFIDENTIAL", right_text="Exhibit [x]", center_text="Abira Medical Laboratories v. Comtron Corp.", exhibit_title="Summary of X"):
    packet = io.BytesIO()
    can = canvas.Canvas(packet, pagesize=(page_width, page_height))

    can.setFont("Times-Roman", 11) # Set font: Times New Roman (regular), size 11
    can.setFillColorRGB(0.5, 0.5, 0.5)  # RGB for gray

    # First line: Alpha (left) and Bravo (right)
    y_top = page_height - 30
    can.drawString(40, y_top, left_text)
    can.drawRightString(page_width - 40, y_top, right_text)

    # Second line: Charlie (centered, just below)
    y_center = y_top - 15  # 15 points below first line
    can.drawCentredString(page_width / 2, y_center, center_text)

    # Third line: Titla (centered, just below)
    y_third = y_center - 15  # 15 points below first line
    can.drawCentredString(page_width / 2, y_third, exhibit_title)

    #Find the bottom of the printable area
    y_bottom = 612-100 - 15
    can.line(40, y_bottom, page_width - 40, y_bottom)


    can.save()
    packet.seek(0)
    return PdfReader(packet)

# Read your existing PDF
reader = PdfReader(r"C:\Users\Matthew.Vaughn\Desktop\Mishandled Claims Test.pdf")
writer = PdfWriter()

for page in reader.pages:
    page_width = float(page.mediabox.width)
    page_height = float(page.mediabox.height)

    # Create header overlay PDF
    header_pdf = create_header(page_width, page_height)
    header_page = header_pdf.pages[0]

    # Merge header onto original page
    page.merge_page(header_page)
    writer.add_page(page)

# Save output
with open(r"C:\Users\Matthew.Vaughn\Desktop\Mishandled Claims Test Output.pdf", "wb") as f:
    writer.write(f)

In [14]:
from PyPDF2 import PdfMerger

pdfs = [r"\\wdcfs1\dcdata6\Cases\Active\Abira Medical (102312)\06 Workstreams\0022 Overall Damages Summary Exhibit\0022 Overall Damages Summary.pdf", 
        r"\\wdcfs1\dcdata6\Cases\Active\Abira Medical (102312)\06 Workstreams\0023 Genesis Profit and Loss Exhibit\0023 Genesis Profit and Loss Exhibit.pdf", 
        r"\\wdcfs1\dcdata6\Cases\Active\Abira Medical (102312)\06 Workstreams\0010 Mishandled Claims Damages\0010 Mishandled Claims Damages.pdf",
        r"\\wdcfs1\dcdata6\Cases\Active\Abira Medical (102312)\06 Workstreams\0013 Unbilled Claims Damages\0013 Unbilled Claims Damages.pdf",
        r"\\wdcfs1\dcdata6\Cases\Active\Abira Medical (102312)\06 Workstreams\0016 Lost Client Damages\0016 Lost Client Damages.pdf",
        r"\\wdcfs1\dcdata6\Cases\Active\Abira Medical (102312)\06 Workstreams\0024 Actual vs ButFor Revenue\0024 Actual vs ButFor Revenue.pdf",
        r"\\wdcfs1\dcdata6\Cases\Active\Abira Medical (102312)\06 Workstreams\0021 Realization Rates Exhibit\0021 Realization Rates.pdf",
        r"\\wdcfs1\dcdata6\Cases\Active\Abira Medical (102312)\06 Workstreams\0017 Patient Responsibility Damages\0017 Patient Responsibility Damages.pdf"]

merger = PdfMerger()

for pdf in pdfs:
    merger.append(pdf)

merger.write(r"\\wdcfs1\dcdata6\Cases\Active\Abira Medical (102312)\08 Report\Draft\Exhibits\Exhibits.pdf")
merger.close()